# 01 · Data collection

Race results, qualifying and sprint classifications for every Formula 1 Grand Prix from
**2014** (start of the V6 turbo-hybrid era) to the latest completed round of **2026**, from the
[Jolpica-F1 API](https://github.com/jolpica/jolpica-f1), the community-run successor to Ergast.

The client in `f1predict.data` throttles to one request per second, retries on HTTP 429/5xx
and caches every raw JSON page under `data/raw/jolpica/` (git-ignored), so re-running this
notebook does not hit the API again. The tidy tables it writes to `data/processed/` *are*
committed, so everything downstream runs without network access.

In [1]:
import pandas as pd

from f1predict.data import build_dataset

pd.set_option("display.max_columns", 30)

tables = build_dataset(start=2014, end=2026, refresh_current=False)
{name: df.shape for name, df in tables.items()}

2014: 407 results, 405 quali rows
2015: 378 results, 374 quali rows


2016: 462 results, 457 quali rows
2017: 400 results, 398 quali rows


2018: 420 results, 420 quali rows


2019: 420 results, 418 quali rows
2020: 340 results, 340 quali rows


2021: 440 results, 439 quali rows


2022: 440 results, 440 quali rows


... (2 outputs left out)

2025: 479 results, 479 quali rows


2026: 440 results, 435 quali rows


D:\motorsport projects\f1-race-outcome-predictor\src\f1predict\data.py:217: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([p for p in parts if not p.empty], ignore_index=True).drop_duplicates()


{'races': (272, 10),
 'results': (5545, 17),
 'qualifying': (5524, 11),
 'sprints': (612, 17)}

## What we have

In [2]:
races = tables["races"]
races.groupby("season").size().rename("races").to_frame().T

season,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
races,19,19,21,20,21,21,17,22,22,22,24,24,20


In [3]:
tables["results"].head()

,season,round,driver_id,driver_code,driver_name,constructor_id,constructor_name,number,grid,position,position_text,points,laps,status,time_millis,fastest_lap_rank,fastest_lap_time
0,2014,1,rosberg,ROS,Nico Rosberg,mercedes,Mercedes,6,3,1,1,25.0,57,Finished,5578710.0,1.0,1:32.478
1,2014,1,kevin_magnussen,MAG,Kevin Magnussen,mclaren,McLaren,20,4,2,2,18.0,57,Finished,5605487.0,6.0,1:33.066
2,2014,1,button,BUT,Jenson Button,mclaren,McLaren,22,10,3,3,15.0,57,Finished,5608737.0,5.0,1:32.917
3,2014,1,alonso,ALO,Fernando Alonso,ferrari,Ferrari,14,5,4,4,12.0,57,Finished,5613994.0,7.0,1:33.186
4,2014,1,bottas,BOT,Valtteri Bottas,williams,Williams,77,15,5,5,10.0,57,Finished,5626349.0,3.0,1:32.616


In [4]:
tables["qualifying"].head()

,season,round,driver_id,driver_code,driver_name,constructor_id,constructor_name,quali_position,q1,q2,q3
0,2014,1,hamilton,HAM,Lewis Hamilton,mercedes,Mercedes,1,1:31.699,1:42.890,1:44.231
1,2014,1,ricciardo,RIC,Daniel Ricciardo,red_bull,Red Bull,2,1:30.775,1:42.295,1:44.548
2,2014,1,rosberg,ROS,Nico Rosberg,mercedes,Mercedes,3,1:32.564,1:42.264,1:44.595
3,2014,1,kevin_magnussen,MAG,Kevin Magnussen,mclaren,McLaren,4,1:30.949,1:43.247,1:45.745
4,2014,1,alonso,ALO,Fernando Alonso,ferrari,Ferrari,5,1:31.388,1:42.805,1:45.819


## Data quality checks

* every race should have results and qualifying,
* a handful of drivers have no qualifying time (crashed in Q1, excluded, etc.) — these become
  missing values, which LightGBM handles natively,
* `grid == 0` means a pit-lane start.

In [5]:
res, quali = tables["results"], tables["qualifying"]
checks = pd.DataFrame(
    {
        "races": races.groupby("season").size(),
        "results rows": res.groupby("season").size(),
        "quali rows": quali.groupby("season").size(),
        "pit-lane starts": res[res["grid"] == 0].groupby("season").size(),
        "sprints": tables["sprints"].groupby("season")["round"].nunique(),
    }
).fillna(0).astype(int)
checks

,races,results rows,quali rows,pit-lane starts,sprints
season,,,,,
2014,19,407,405,0,0
2015,19,378,374,3,0
2016,21,462,457,4,0
2017,20,400,398,2,0
2018,21,420,420,1,0
2019,21,420,418,12,0
2020,17,340,340,5,0
2021,22,440,439,12,3
2022,22,440,440,11,3


In [6]:
missing_quali = res.merge(quali[["season", "round", "driver_id"]], how="left", indicator=True,
                          on=["season", "round", "driver_id"])
print(f"Results without a qualifying row: {(missing_quali['_merge'] == 'left_only').sum()}")

Results without a qualifying row: 22


## Finishing status

Jolpica changed how it labels lapped / retired cars in recent seasons (e.g. `Lapped`,
`Retired`), so classification logic in `features.is_finished` accepts both the old
(`+1 Lap`) and new forms.

In [7]:
res["status"].value_counts().head(15)

status
Finished            2956
+1 Lap               991
Lapped               426
Retired              271
+2 Laps              188
Collision            129
Accident              70
Engine                66
Collision damage      46
Brakes                45
Power Unit            41
Gearbox               37
Suspension            25
+3 Laps               24
Disqualified          17
Name: count, dtype: int64